In [1]:
import cv2
import numpy as np
import os

print("Everything is working")

Everything is working


In [2]:
import os

print(os.listdir("input_images"))

['round_01.jpeg', 'round_02.jpeg', 'round_03.jpeg', 'round_04.jpeg', 'round_05.jpeg', 'round_06.jpeg', 'round_07.jpeg', 'square_01.jpeg', 'square_02.jpeg', 'square_03.jpeg', 'square_04.jpeg', 'square_05.jpeg', 'square_06.jpeg', 'square_07.jpeg']


In [3]:
import cv2
import numpy as np
import os

input_folder = "input_images"
output_folder = "output_images"

os.makedirs(output_folder, exist_ok=True)

for filename in os.listdir(input_folder):

    if not filename.lower().endswith((".jpg", ".jpeg", ".png")):
        continue

    image_path = os.path.join(input_folder, filename)
    image = cv2.imread(image_path)

    if image is None:
        print("Could not read:", filename)
        continue

    image = cv2.resize(image, (600, 800))
    output = image.copy()

    # Convert to HSV
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

    # Biscuit color range
    lower_biscuit = np.array([5, 35, 50])
    upper_biscuit = np.array([35, 255, 255])

    mask = cv2.inRange(hsv, lower_biscuit, upper_biscuit)

    # Clean mask
    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    for contour in contours:

        area = cv2.contourArea(contour)

        if area < 700:
            continue

        x, y, w, h = cv2.boundingRect(contour)

        perimeter = cv2.arcLength(contour, True)
        if perimeter == 0:
            continue

        circularity = 4 * np.pi * area / (perimeter * perimeter)

        hull = cv2.convexHull(contour)
        hull_area = cv2.contourArea(hull)

        if hull_area == 0:
            continue

        solidity = area / hull_area

        rect_area = w * h
        extent = area / rect_area

        name = filename.lower()

        # ---------------- ROUND BISCUIT ----------------
        if name.startswith("round"):

            if area > 12000 and circularity > 0.68 and solidity > 0.92:
                label = "Intact Biscuit"
                color = (0, 255, 0)
            else:
                label = "Broken Biscuit"
                color = (0, 0, 255)

        # ---------------- SQUARE BISCUIT ----------------
        elif name.startswith("square"):

            approx = cv2.approxPolyDP(contour, 0.04 * perimeter, True)
            corners = len(approx)

            if area > 9000 and extent > 0.68 and solidity > 0.93 and 4 <= corners <= 6:
                label = "Intact Biscuit"
                color = (0, 255, 0)
            else:
                label = "Broken Biscuit"
                color = (0, 0, 255)

        # ---------------- DEFAULT ----------------
        else:
            label = "Biscuit"
            color = (255, 0, 0)

        cv2.rectangle(output, (x, y), (x + w, y + h), color, 2)

        cv2.putText(
            output,
            label,
            (x, y - 8),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            color,
            2
        )

    output_path = os.path.join(output_folder, "result_" + filename)
    cv2.imwrite(output_path, output)

    print("Saved:", output_path)

print("Finished processing all images.")

Saved: output_images\result_round_01.jpeg
Saved: output_images\result_round_02.jpeg
Saved: output_images\result_round_03.jpeg
Saved: output_images\result_round_04.jpeg
Saved: output_images\result_round_05.jpeg
Saved: output_images\result_round_06.jpeg
Saved: output_images\result_round_07.jpeg
Saved: output_images\result_square_01.jpeg
Saved: output_images\result_square_02.jpeg
Saved: output_images\result_square_03.jpeg
Saved: output_images\result_square_04.jpeg
Saved: output_images\result_square_05.jpeg
Saved: output_images\result_square_06.jpeg
Saved: output_images\result_square_07.jpeg
Finished processing all images.
